# Tree Biomass from Field Measurements

This tutorial uses the `miaproc.biomass` API to estimate direct tree biomass for adult mangrove trees from field measurements.

The fixture is copied from `miaproc/tests/data/biomass/20260303_biomass_test.csv`. It intentionally uses field-style column names, so the tutorial also demonstrates `BiomassColumns`.


In [ ]:
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT.name != "mia-tutorials" and ROOT.parent != ROOT:
    ROOT = ROOT.parent

DATA = ROOT / "data"
REPOS = ROOT.parent
ROOT

import pandas as pd
from miaproc.biomass import BiomassColumns, estimate_trees, load_packaged_equations

biomass_csv = DATA / "miaproc_biomass" / "20260303_biomass_test.csv"
raw = pd.read_csv(biomass_csv)
raw.shape, raw.columns.tolist()[:8]


## Inspect the source table

The biomass API only needs species, DBH, height, and life-stage fields for this tutorial. The source table has more site and plot metadata that can be carried through unchanged.


In [ ]:
source_cols = ["Species", "DBH (cm)", "Total Height (m)", "Juvenil / Adult", "condition_live_or_dead"]
raw[source_cols].head(12)


In [ ]:
raw["Juvenil / Adult"].value_counts(dropna=False)


## Load packaged equations

`miaproc` ships a unified equation table. For direct mangrove biomass, use `dataset="dina"`; this selects the four mangrove direct-biomass equations with wood density already substituted in `equation_numpy_wd_fixed`.


In [ ]:
equations = load_packaged_equations()

dina = equations[equations["source_dataset"].eq("dina")]
dina[[
    "source_record_id",
    "scientific_name_apg_raw",
    "response_variable",
    "response_units",
    "equation_numpy_wd_fixed",
]]


## Map field columns and estimate biomass

The package defaults are `species`, `dbh_cm`, `tree_height_m`, and `life_stage`. This fixture uses legacy field labels, so we map them explicitly.


In [ ]:
cols = BiomassColumns(
    species="Species",
    dbh_cm="DBH (cm)",
    height_m="Total Height (m)",
    life_stage="Juvenil / Adult",
)

tutorial_rows = raw[source_cols].copy()
out = estimate_trees(tutorial_rows, equations=equations, columns=cols, dataset="dina")

summary_cols = [
    "Species",
    "DBH (cm)",
    "Juvenil / Adult",
    "estimate_response_variable",
    "response_units",
    "match_status",
    "range_status",
    "source_record_id",
]
out[summary_cols].head(20)


## Interpret status fields

The status columns are as important as the numeric estimate: they explain whether a row was evaluated, skipped for missing DBH, excluded by life stage, or matched through the allometry table.


In [ ]:
status_summary = (
    out.groupby(["match_status", "range_status"], dropna=False)
    .size()
    .reset_index(name="rows")
    .sort_values("rows", ascending=False)
)
status_summary


In [ ]:
success = out[out["estimate_response_variable"].notna()].copy()
success[["Species", "DBH (cm)", "estimate_response_variable", "source_record_id", "equation_code"]].head(10)


## Plot a quick sanity check

For a tutorial, the goal is not scientific validation yet; it is to make sure the API contract is visible and inspectable.


In [ ]:
ax = success.plot.scatter(
    x="DBH (cm)",
    y="estimate_response_variable",
    title="Direct biomass estimates from packaged dina equations",
)
ax.set_ylabel("Biomass estimate (kg)");
